In [54]:
"""
Penerapannya di scikit-learn ada bebebrapa:
    KNeighborsClassifier
    KNeighborsRegression
    KNeighborsTransformer
    NearestNeighbors
"""

'\nPenerapannya di scikit-learn ada bebebrapa:\n    KNeighborsClassifier\n    KNeighborsRegression\n    KNeighborsTransformer\n    NearestNeighbors\n'

In [55]:
"""
Parameter utama:
    n_neighbors: Jumlah Tetangga (K)
    metrics: Metric Jarak           -> default=minkowski ;opsi=euclidean,manhattan,minkowski,cosine similarity
    weight: Bobot (Weights)         -> default=Uniform   ;opsi=uniform,distance             -> uniform=>semua tetangga punya bobot sama; distance=>bobot tergantung jarak
    algorithm: Algoritma Pencarian Tetangga -> default=auto ;opsi:ball_tree,kd_tree,brute,auto
    Panjang Jarak (Distance Metric Parameters)

Parameter Lainnya di Scikit-Learn:
    n_jobs: jumlah core COU yg dipake                       -> default=None (1core) ;opsi:number, -1 untuk sejumlah semua core
    p:  paramemer power untuk metrik Manhattan Distance     -> default=2 ;opsi:1,2 (1=manhattan,2=eucladean)
    leaf_size: Ukuran daun untuk BallTree atau KDTree       -> default=30           ;Mempengaruhi kecepatan build & query, serta memori
    metric_params: Parameter tambahan untuk metric kustom   -> default=None         ;Contoh: {'w': [1, 2, 3]} untuk weighted Minkowski
"""

"\nParameter utama:\n    n_neighbors: Jumlah Tetangga (K)\n    metrics: Metric Jarak           -> default=minkowski ;opsi=euclidean,manhattan,minkowski,cosine similarity\n    weight: Bobot (Weights)         -> default=Uniform   ;opsi=uniform,distance             -> uniform=>semua tetangga punya bobot sama; distance=>bobot tergantung jarak\n    algorithm: Algoritma Pencarian Tetangga -> default=auto ;opsi:ball_tree,kd_tree,brute,auto\n    Panjang Jarak (Distance Metric Parameters)\n\nParameter Lainnya di Scikit-Learn:\n    n_jobs: jumlah core COU yg dipake                       -> default=None (1core) ;opsi:number, -1 untuk sejumlah semua core\n    p:  paramemer power untuk metrik Manhattan Distance     -> default=2 ;opsi:1,2 (1=manhattan,2=eucladean)\n    leaf_size: Ukuran daun untuk BallTree atau KDTree       -> default=30           ;Mempengaruhi kecepatan build & query, serta memori\n    metric_params: Parameter tambahan untuk metric kustom   -> default=None         ;Contoh: {'w': [1

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import LabelEncoder
import pandas as pd
import numpy as np

In [57]:
df = pd.DataFrame({
    "berat_badan": [55, 53, 48, 90, 65, 70, 45, 80, 60, 58],
    "tinggi": [160, 175, 152, 182, 168, 178, 155, 185, 163, 170],
    "gender": ["P", "L", "P", "L", "L", "L", "P", "L", "P", "P"]
})
x = df.drop(columns=['gender'])
y = df['gender']

model = KNeighborsClassifier(
    n_neighbors=3,
    weights='distance',
    metric='minkowski',
    p=1,
    n_jobs=-1
)
model.fit(x, y)
model.predict(
    pd.DataFrame({
        "berat_badan": [49],
        "tinggi": [170]
    })
)

array(['P'], dtype=object)

In [58]:
hyper_params = {
    "weights": ["distance", "uniform"],
    "metric": ["euclidean", "manhattan", "cosine"],
    "n_neighbors": [1, 2, 3],

}

model = KNeighborsClassifier(n_jobs=2)
fine_tuned = GridSearchCV(estimator=model,
                          scoring='accuracy',
                          cv=3,
                          param_grid=hyper_params)

fine_tuned.fit(x, y)

,estimator,KNeighborsClassifier(n_jobs=2)
,param_grid,"{'metric': ['euclidean', 'manhattan', ...], 'n_neighbors': [1, 2, ...], 'weights': ['distance', 'uniform']}"
,scoring,'accuracy'
,n_jobs,None
,refit,True
,cv,3
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_neighbors,2


In [59]:
preds = fine_tuned.predict(x)
accuracy_score(y, preds)

0.9

In [60]:
fine_tuned.predict(pd.DataFrame({
    "berat_badan": [32],
    "tinggi": [120]
}))

array(['P'], dtype=object)

### Menggunakan KNN untuk klasifikasi teks kalimat

In [3]:
df = pd.DataFrame({
    "kalimat": [
        "tolong bantu saya, saya sedang punya masalah",
        "saya butuh pertolongan",
        "harga produk ini berapa",
        "berapa biaya pengiriman",
        "terima kasih banyak",
        "makasih ya sudah membantu",
        "mau tanya harga dong",
        "cara pembayaran gimana",
        "oke siap terima kasih",
        "tolong info harganya",
    ],
    "kategori": ["darurat", "darurat", "harga", "harga", "pujian", "pujian", "harga", "harga", "pujian", "harga"]
})

from sklearn.pipeline import FeatureUnion
# metric cosine tidak bisa label angka, jadi target harus di encode
encoder = LabelEncoder()
vectorizer_char = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 4))
vectorizer_word = TfidfVectorizer(ngram_range=(1, 2))
vectorizer = FeatureUnion([
    ('word', vectorizer_word),
    ('char', vectorizer_char)
])
x = vectorizer.fit_transform(df['kalimat'])
y = encoder.fit_transform(df["kategori"])
print(x.shape)
print(x[:5])
print(pd.DataFrame(
    data=x[:5].toarray(),
    columns=vectorizer.get_feature_names_out()
))

params = {
    "n_neighbors": [2, 3, 4],
    "weights": ["uniform", "distance"],
    "metric": ["cosine"] # ["euclidean", "manhattan", "cosine"],
}

model = KNeighborsClassifier(n_jobs=-1)
# model_tuned = RandomizedSearchCV(estimator=model,
#                                  param_distributions=params,
#                                  scoring='accuracy',
#                                  cv=2,
#                                  n_iter=50,
#                                  n_jobs=-1,
#                                  random_state=42)
model_tuned = GridSearchCV(estimator=model,
                           param_grid=params,
                           cv=5,
                           n_jobs=-1,
                           scoring='accuracy',
                           verbose=2)
model_tuned.fit(x, y)

(10, 308)
<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 239 stored elements and shape (5, 308)>
  Coords	Values
  (0, 53)	0.2304009512288975
  (0, 0)	0.2710308115997323
  (0, 39)	0.460801902457795
  (0, 43)	0.2710308115997323
  (0, 37)	0.2710308115997323
  (0, 25)	0.2710308115997323
  (0, 54)	0.2710308115997323
  (0, 1)	0.2710308115997323
  (0, 41)	0.2710308115997323
  (0, 42)	0.2710308115997323
  (0, 44)	0.2710308115997323
  (0, 38)	0.2710308115997323
  (0, 107)	0.10745614348231283
  (0, 288)	0.09401142777396732
  (0, 242)	0.09401142777396732
  (0, 207)	0.09401142777396732
  (0, 244)	0.08358289608270937
  (0, 227)	0.18802285554793463
  (0, 108)	0.10745614348231283
  (0, 289)	0.09401142777396732
  (0, 243)	0.09401142777396732
  (0, 208)	0.09401142777396732
  (0, 245)	0.09401142777396732
  (0, 58)	0.10745614348231283
  (0, 144)	0.09401142777396732
  :	:
  (4, 267)	0.16158904088894194
  (4, 105)	0.18469813270658197
  (4, 286)	0.18469813270658197
  (4, 173)	0.1846981327065

/media/ujun/d/latihan_ngoding/Machine Learning/.310venv/lib/python3.10/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(


[CV] END ......metric=cosine, n_neighbors=2, weights=uniform; total time=   0.0s
[CV] END .....metric=cosine, n_neighbors=2, weights=distance; total time=   0.1s
[CV] END .....metric=cosine, n_neighbors=2, weights=distance; total time=   0.0s[CV] END ......metric=cosine, n_neighbors=2, weights=uniform; total time=   0.1s

[CV] END ......metric=cosine, n_neighbors=2, weights=uniform; total time=   0.0s
[CV] END ......metric=cosine, n_neighbors=2, weights=uniform; total time=   0.1s
[CV] END ......metric=cosine, n_neighbors=2, weights=uniform; total time=   0.1s
[CV] END .....metric=cosine, n_neighbors=2, weights=distance; total time=   0.1s
[CV] END .....metric=cosine, n_neighbors=2, weights=distance; total time=   0.1s
[CV] END .....metric=cosine, n_neighbors=2, weights=distance; total time=   0.1s
[CV] END ......metric=cosine, n_neighbors=3, weights=uniform; total time=   0.1s
[CV] END ......metric=cosine, n_neighbors=3, weights=uniform; total time=   0.1s
[CV] END ......metric=cosine

,estimator,KNeighborsCla...ier(n_jobs=-1)
,param_grid,"{'metric': ['cosine'], 'n_neighbors': [2, 3, ...], 'weights': ['uniform', 'distance']}"
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,5
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_neighbors,2


In [7]:
model_tuned.best_params_

{'metric': 'cosine', 'n_neighbors': 2, 'weights': 'distance'}

In [4]:
result = model_tuned.predict(vectorizer.transform(["aku perlu pertolongan ?"]))
encoder.inverse_transform(result)

array(['darurat'], dtype=object)

# NearestNeighbors
Bisa dipake untuk deteksi anomali

In [105]:
# data normal
data = pd.DataFrame({
    "Angka": [4, 6, 2, 6, 12, 3, 2, 15, 20, 18, 21, 15]  
})

model =  NearestNeighbors(n_neighbors=3, n_jobs=2)
model.fit(data)

distances, _ = model.kneighbors(pd.DataFrame({"Angka": [100]}))
np.average(distances)

np.float64(80.33333333333333)